# Verify the video timing QC change in the batch pipeline

The library branch `plan/video-timing-qc` replaces the timing QC inside
`integrate_keypoints_with_video_time`: keypoint `time_raw` now comes from
Harp time corrected for dropped frames and Harp glitches, and CSVs with a
header row load instead of crashing. This notebook runs `run_batch_analysis`
on a few attached sessions twice, once with library `main` and once with the
branch, and diffs every intermediate parquet, to catch anything that changes
unexpectedly.

Steps:

1. Clone both library versions into scratch. Each run imports its version
   through `PYTHONPATH` in a subprocess, so the capsule's installed library
   is not touched.
2. Build the prediction-CSV list for every attached session with raw data
   and a `videoprocessed` asset (saved to scratch).
3. Classify every session's bottom-camera timing with `video_timing_qc`, and
   pick a few sessions to run: up to 2 per problem class (glitch, drops,
   ...), up to 2 header-row CSVs, and 2 ok ones. The scan reads every
   session's CSV; only the chosen few go through the batch.
4. Run the batch with each version, then diff (`scripts/compare_batch_outputs.py`).

What to expect:

| Timing class | Expected diff |
|---|---|
| `ok` | every file `identical` |
| `harp_glitch` | tiny differences in time columns (the old QC also interpolated glitches, and nudged the next row by µs) |
| header-row CSV | `missing in old`: `main` crashes on these files |
| `frame_drops` | large differences in time and everything matched by time |

None of the attached sessions has dropped frames (checked from the CSVs on
S3, 2026-09-29), so the drop case is covered by the library's
`examples/video_timing_qc_validation.ipynb` instead.

Everything is written under `/root/capsule/scratch/video_timing_qc_test/`.
Code-Ocean-only, guarded by `IS_CO`. Run top to bottom in a fresh kernel.

## 1. Setup

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

if Path("/root/capsule").exists():
    DATA = Path("/root/capsule/data")
    WORK = Path("/root/capsule/scratch/video_timing_qc_test")
    WORK.mkdir(parents=True, exist_ok=True)
    IS_CO = True
else:
    DATA = WORK = None
    IS_CO = False
    print("Local environment: nothing to run here. Run this notebook on Code Ocean.")

LIBRARY_URL = "https://github.com/AllenNeuralDynamics/aind-dynamic-foraging-behavior-video-analysis.git"
OLD_REF = "main"                  # library before the change
NEW_REF = "plan/video-timing-qc"  # library with video_timing_qc

N_OK = 2            # ok sessions to run
N_PER_CLASS = 2     # max sessions per other class (glitch, header row, ...)
RUN_TRIGGER_LOG = False  # also run the branch with use_trigger_log=True

## 2. Clone both library versions

Shallow clones into scratch. Delete a folder to re-clone it (e.g. after new
commits on the branch).

In [ ]:
def clone(ref, folder):
    """Shallow-clone the library at `ref` into `folder`, once; return its commit."""
    if not folder.exists():
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1", "--branch", ref, LIBRARY_URL, str(folder)],
            check=True,
        )
    return subprocess.run(
        ["git", "-C", str(folder), "rev-parse", "--short", "HEAD"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()


if IS_CO:
    LIB_OLD = WORK / "lib_old"
    LIB_NEW = WORK / "lib_new"
    print(f"old: {OLD_REF} @ {clone(OLD_REF, LIB_OLD)}")
    print(f"new: {NEW_REF} @ {clone(NEW_REF, LIB_NEW)}")
    assert not (LIB_OLD / "src/aind_dynamic_foraging_behavior_video_analysis/video_timing_qc.py").exists(), \
        "old clone already has video_timing_qc -- has the branch been merged?"
    assert (LIB_NEW / "src/aind_dynamic_foraging_behavior_video_analysis/video_timing_qc.py").exists()

The rest of this kernel uses the **new** library (for `video_timing_qc` and
the session lookup helpers). It must come first on `sys.path` before anything
imports the library, hence the fresh kernel.

In [ ]:
if IS_CO:
    sys.path.insert(0, str(LIB_NEW / "src"))
    import aind_dynamic_foraging_behavior_video_analysis as lib
    from aind_dynamic_foraging_behavior_video_analysis import video_timing_qc as vtq
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
        find_behavior_videos_folder,
        find_video_csv_path,
    )

    assert Path(lib.__file__).is_relative_to(LIB_NEW), lib.__file__
    print("kernel library:", lib.__file__)

## 3. Prediction CSVs for every attached session

One per session: the first `pred_outputs/video_preds/*predictions.csv` across
that session's `videoprocessed` assets. Which model it is does not matter
here, because both runs use the same file. The full list is saved to
`pred_csv_list.json`.

In [ ]:
if IS_CO:
    raw_sessions = sorted(
        p.name for p in DATA.iterdir()
        if p.is_dir() and p.name.startswith("behavior_") and p.name.count("_") == 3
    )
    pred_csvs = {}
    for session in raw_sessions:
        candidates = sorted(DATA.glob(f"{session}_videoprocessed*/pred_outputs/video_preds/*predictions.csv"))
        if candidates:
            pred_csvs[session] = str(candidates[0])

    (WORK / "pred_csv_list.json").write_text(json.dumps(list(pred_csvs.values()), indent=2))
    print(f"{len(raw_sessions)} raw sessions attached, {len(pred_csvs)} with predictions")
    print("saved", WORK / "pred_csv_list.json")

## 4. Classify each session's video timing

Runs `check_video_timing` on the video CSV the pipeline uses
(`find_video_csv_path`, bottom camera). Takes a few seconds per session;
cached in `timing_scan.csv`.

In [ ]:
if IS_CO:
    scan_path = WORK / "timing_scan.csv"
    if scan_path.exists():
        scan = pd.read_csv(scan_path)
    else:
        rows = []
        for session in pred_csvs:
            videos = find_behavior_videos_folder(str(DATA / session))
            csv_path = find_video_csv_path(videos) if videos else None
            row = {"session": session, "csv_path": str(csv_path)}
            if csv_path is None:
                row["qc_class"] = "no_video_csv"
            else:
                with open(csv_path) as f:
                    row["has_header"] = not f.readline()[0].isdigit()
                try:
                    qc = vtq.check_video_timing(vtq.load_video_timing(csv_path))
                    row.update({k: qc[k] for k in [
                        "qc_class", "n_rows", "n_frames_dropped", "glitch_rows",
                        "n_unexplained_flags", "clock_slip_frames",
                    ]})
                except ValueError as e:
                    row.update({"qc_class": "unreadable", "error": str(e)})
            rows.append(row)
            print(session, row["qc_class"], "(header)" if row.get("has_header") else "")
        scan = pd.DataFrame(rows)
        scan.to_csv(scan_path, index=False)
    print(scan["qc_class"].value_counts().to_string())
    print(f"header-row CSVs: {int(scan['has_header'].fillna(False).sum())}")

## 5. Choose sessions

Every class other than `ok` (up to `N_PER_CLASS` each), header-row CSVs (up
to `N_PER_CLASS`), and `N_OK` ok sessions.

In [ ]:
if IS_CO:
    header = scan["has_header"].fillna(False).astype(bool)
    chosen = pd.concat([
        scan[~header & (scan["qc_class"] != "ok")].groupby("qc_class").head(N_PER_CLASS),
        scan[header].head(N_PER_CLASS),
        scan[~header & (scan["qc_class"] == "ok")].head(N_OK),
    ])
    chosen = chosen.assign(
        group=lambda d: d["qc_class"].where(~d["has_header"].fillna(False).astype(bool), "header_row")
    )
    selected_preds = [pred_csvs[s] for s in chosen["session"]]
    (WORK / "pred_csv_selected.json").write_text(json.dumps(selected_preds, indent=2))
    print(chosen[["session", "group", "qc_class", "glitch_rows", "n_frames_dropped"]].to_string(index=False))

## 6. Run the batch with each library version

Each run is a subprocess with that clone's `src` first on `PYTHONPATH`; it
checks which library it imported before running. Output goes to
`out_old/` and `out_new/` (and `out_log/` if `RUN_TRIGGER_LOG`), the log of
each run to `<name>.log`. `force_rerun=True`, clips off. Allow a few minutes
per session per run.

In [ ]:
RUNNER = """
import json, sys
from pathlib import Path

lib_src, out_dir, preds_json, data_root, use_log = sys.argv[1:6]
import aind_dynamic_foraging_behavior_video_analysis as lib
assert Path(lib.__file__).resolve().is_relative_to(Path(lib_src).resolve()), lib.__file__
print("library:", lib.__file__, flush=True)

from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_analysis import run_batch_analysis
kwargs = {"use_trigger_log": True} if use_log == "1" else {}
run_batch_analysis(
    json.loads(Path(preds_json).read_text()), Path(data_root), out_dir,
    extract_clips=False, force_rerun=True, **kwargs,
)
"""


def run_batch(name, lib_dir, use_trigger_log=False):
    """Run run_batch_analysis on the selected sessions with one library version."""
    runner = WORK / "run_batch.py"
    runner.write_text(RUNNER)
    env = {**os.environ, "PYTHONPATH": str(lib_dir / "src"), "MPLBACKEND": "Agg"}
    log_path = WORK / f"{name}.log"
    with open(log_path, "w") as log:
        result = subprocess.run(
            [sys.executable, str(runner), str(lib_dir / "src"), str(WORK / name),
             str(WORK / "pred_csv_selected.json"), str(DATA), "1" if use_trigger_log else "0"],
            env=env, stdout=log, stderr=subprocess.STDOUT,
        )
    tail = log_path.read_text().splitlines()
    print(f"{name}: exit {result.returncode}; library line: {tail[0] if tail else ''}")
    if result.returncode:
        print("\n".join(tail[-30:]))

In [ ]:
if IS_CO:
    run_batch("out_old", LIB_OLD)
    run_batch("out_new", LIB_NEW)
    if RUN_TRIGGER_LOG:
        run_batch("out_log", LIB_NEW, use_trigger_log=True)

## 7. Compare

`compare_batch_outputs.py` prints, per session and parquet, `identical` or
what changed, and each run's `batch_error_log.txt`.

In [ ]:
def compare(old, new):
    """Run the branch's compare script; return its per-file table."""
    table_path = WORK / f"diff_{old}_vs_{new}.csv"
    result = subprocess.run(
        [sys.executable, str(LIB_NEW / "scripts/compare_batch_outputs.py"),
         str(WORK / old), str(WORK / new), "--csv", str(table_path)],
        capture_output=True, text=True,
    )
    print(result.stdout, result.stderr)
    return pd.read_csv(table_path)


if IS_CO:
    diff = compare("out_old", "out_new")

Per session: the timing class, how many files were identical, and whether
that matches the expectation table at the top. `ok` sessions that are not
all-identical are the ones to look at first.

In [ ]:
def expected(group):
    return {
        "ok": "all identical",
        "harp_glitch": "time columns differ slightly",
        "header_row": "missing in old",
        "frame_drops": "large time differences",
    }.get(group, "review")


if IS_CO:
    per_session = (
        diff.assign(identical=diff["result"].eq("identical"))
        .groupby("session")
        .agg(files=("file", "size"), identical=("identical", "sum"),
             missing_in_old=("result", lambda r: r.eq("missing in old").sum()),
             missing_in_new=("result", lambda r: r.eq("missing in new").sum()))
        .reset_index()
        .merge(chosen[["session", "group"]], on="session", how="right")
    )
    per_session["expected"] = per_session["group"].map(expected)
    per_session["flag"] = (
        ((per_session["group"] == "ok") & (per_session["identical"] != per_session["files"]))
        | ((per_session["group"] != "header_row") & (per_session["missing_in_new"] > 0))
        | per_session["files"].isna()
    ).map({True: "CHECK", False: ""})
    print(per_session.to_string(index=False))

In [ ]:
if IS_CO and RUN_TRIGGER_LOG:
    # CSV vs trigger-log correction: expected identical (no drops -> no estimated rows)
    diff_log = compare("out_new", "out_log")

## 8. Summary

- All `ok` sessions `identical` and no `CHECK` flags: the change only
  affects sessions whose timing needed correcting.
- Header-row sessions present only in `out_new`: the new-layout fix works.
- Glitch sessions: confirm the differing columns are time columns and the
  changes are small (µs to ~1 s on a handful of rows).

Record the outcome in the library's `VIDEO_TIMING_QC_PLAN.md` (Phase 3).